In [1]:
from dotenv import load_dotenv
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

load_dotenv()

# Verificar claves API críticas
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY no encontrada en .env")
if not os.getenv("LANGSMITH_API_KEY"):
    raise ValueError("LANGSMITH_API_KEY no encontrada en .env")

In [2]:
from langchain_core.prompts import ChatPromptTemplate

LEGAL_JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
Eres un experto legal especializado en legislación boliviana.

Recibirás:

1. Pregunta legal del usuario.
2. Respuesta de referencia (base de verdad).
3. Respuesta del modelo.

Evalúa la respuesta del modelo de acuerdo con los estándares legales bolivianos.

Verifica:

1. **Exactitud normativa:** ¿coincide con la ley boliviana? (Revisalo únicamente con el contexto de la respuesta de referencia)
2. **Calidad del razonamiento jurídico.**
3. **Presencia de leyes o artículos alucinados (inventados).**
4. **Riesgo de asesoramiento legal engañoso.**

**Calificación:**

* Exactitud Normativa: 1–5
* Razonamiento Jurídico: 1–5
* Citación Correcta: verdadero/falso
* Alucinación Detectada: verdadero/falso

**Nivel de Riesgo:**

* BAJO (LOW)
* MEDIO (MEDIUM)
* ALTO (HIGH)

Devuelve el resultado en formato **JSON**:

```json
{{
  "normative_accuracy": int,
  "legal_reasoning": int,
  "hallucination": true/false,
  "risk_level": "LOW|MEDIUM|HIGH",
  "overall_pass": true/false,
  "justification": "breve explicación"
}}
```
"""),
    ("human", """
Question:
{question}

Reference Answer:
{reference_answer}

Model Answer:
{model_answer}
""")
])


In [3]:
from ai.agents.conversational_assistant.agent import builder 
from langchain_core.messages import HumanMessage


async def rag_legal_model(inputs: dict) -> dict:
    user_input = inputs["question"]

    human_message = HumanMessage(content=user_input)

    asistente = builder.compile()
    result = await asistente.ainvoke(
        {"messages": human_message})

    return {
        "output": result["messages"][-1].content
    }


In [4]:
#await rag_legal_model(inputs={'question':'prueba'})

In [5]:
from pydantic import BaseModel, Field
from typing import Literal

class LegalEvaluation(BaseModel):
    normative_accuracy: int = Field(ge=1, le=5)
    legal_reasoning: int = Field(ge=1, le=5)
    hallucination: bool
    risk_level: Literal["LOW", "MEDIUM", "HIGH"]
    overall_pass: bool
    justification: str


In [6]:
from langchain_openai import ChatOpenAI
judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

structured_llm = judge_llm.with_structured_output(LegalEvaluation)

judge_chain = LEGAL_JUDGE_PROMPT | structured_llm


In [7]:
#structured_judge.invoke([SystemMessage(judge_prompt),HumanMessage("prueba")])

In [ ]:
async def legal_judge_evaluator(inputs, outputs, reference_outputs):
    question = inputs["question"]
    reference = reference_outputs["answer"]
    model_answer = outputs["output"]

    evaluation = await judge_chain.ainvoke({
        "question": question,
        "reference_answer": reference,
        "model_answer": model_answer
    })

    base_score = (
        (evaluation.normative_accuracy * 0.4) +
        (evaluation.legal_reasoning * 0.4) +
        1
    )

    overall_score = (base_score / 5) * 10

    if evaluation.hallucination:
        overall_score -= 1

    if evaluation.risk_level == "HIGH":
        overall_score -= 2
    elif evaluation.risk_level == "MEDIUM":
        overall_score -= 1

    overall_score = round(max(1, min(10, overall_score)), 2)

    return [
        {"key": "overall_score_1_10",  "score": overall_score},
        {"key": "normative_accuracy",  "score": evaluation.normative_accuracy},
        {"key": "legal_reasoning",     "score": evaluation.legal_reasoning},
        {"key": "hallucination",       "score": int(evaluation.hallucination)},
        {"key": "risk_level",          "score": {"LOW": 0, "MEDIUM": 1, "HIGH": 2}[evaluation.risk_level]},
    ]


In [ ]:
from langsmith.evaluation import aevaluate
from langsmith import Client

results = await aevaluate(
    rag_legal_model,
    data='Kantuta AI Evaluation Dataset v6',
    evaluators=[legal_judge_evaluator],
    experiment_prefix="kantuta-ai-eval-v10"
)



View the evaluation results for experiment: 'kantuta-ai-eval-v10-d7c93579' at:
https://smith.langchain.com/o/1e7a3601-ce73-4bbf-8523-f73bb1ff09b4/datasets/0bb9cf2e-1ffb-49f6-96bd-a6b8fc874aa7/compare?selectedSessions=f3e166ff-2909-4f4c-9149-0cfecdad0113




0it [00:00, ?it/s]

content='¿Está permitida la conciliación en casos de violencia contra la mujer?' additional_kwargs={} response_metadata={} id='be6a2f86-39c6-45a8-85d6-1ba3f09dd798'

   Query procesada: '¿Está permitida la conciliación en casos de violencia contra la mujer?'
🔍 [RETRIEVAL] Buscando: '¿Está permitida la conciliación en casos de violencia contra la mujer?'
📡 [RETRIEVAL] Conectando a ChromaDB (Hilo Síncrono)...
   🌍 [RETRIEVAL] Sin filtro de documentos (Búsqueda Global)
📄 [RETRIEVAL] Encontrados 5 documentos.
🤖 [GENERATOR] Iniciando generación...
[SYSTEM PROMPT] Eres **Kantuta AI**, un asistente legal experto en la normativa de Bolivia.

TU OBJETIVO:
Responder a la pregunta del usuario basándote **ÚNICA Y EXCLUSIVAMENTE** en el CONTEXTO LEGAL proporcionado.

ESTILO DE COMNUNICACIÓN
Profesional legal: Explicaciones comprensibles citando si se puede las bases legales.
Empático: Mostrar sensibilidad en temas emocionales, como conflictos familiares o situaciones de violencia.
Neutral y objetiv

/home/huascar/KANTUTA_AI/backend/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=LegalEvaluation(normative...esoramiento engañoso.'), input_type=LegalEvaluation])
  return self.__pydantic_serializer__.to_python(


content='¿Qué es la discriminación racial según la Ley 045?' additional_kwargs={} response_metadata={} id='d3d3a218-c1ad-43d7-9cea-7a81a89df7df'

   Query procesada: '¿Qué es la discriminación racial según la Ley 045?'
🔍 [RETRIEVAL] Buscando: '¿Qué es la discriminación racial según la Ley 045?'
📡 [RETRIEVAL] Conectando a ChromaDB (Hilo Síncrono)...
   🌍 [RETRIEVAL] Sin filtro de documentos (Búsqueda Global)
📄 [RETRIEVAL] Encontrados 5 documentos.
🤖 [GENERATOR] Iniciando generación...
[SYSTEM PROMPT] Eres **Kantuta AI**, un asistente legal experto en la normativa de Bolivia.

TU OBJETIVO:
Responder a la pregunta del usuario basándote **ÚNICA Y EXCLUSIVAMENTE** en el CONTEXTO LEGAL proporcionado.

ESTILO DE COMNUNICACIÓN
Profesional legal: Explicaciones comprensibles citando si se puede las bases legales.
Empático: Mostrar sensibilidad en temas emocionales, como conflictos familiares o situaciones de violencia.
Neutral y objetivo: Evitar juicios personales y centrarse en la legislación a

...

content='¿Qué implica el derecho a la identidad según este Código?' additional_kwargs={} response_metadata={} id='2c89a9cb-503e-4cde-96dd-92a5440fb150'

   Query procesada: '¿Qué implica el derecho a la identidad según este Código?'
🔍 [RETRIEVAL] Buscando: '¿Qué implica el derecho a la identidad según este Código?'
📡 [RETRIEVAL] Conectando a ChromaDB (Hilo Síncrono)...
   🌍 [RETRIEVAL] Sin filtro de documentos (Búsqueda Global)
📄 [RETRIEVAL] Encontrados 5 documentos.
🤖 [GENERATOR] Iniciando generación...
[SYSTEM PROMPT] Eres **Kantuta AI**, un asistente legal experto en la normativa de Bolivia.

TU OBJETIVO:
Responder a la pregunta del usuario basándote **ÚNICA Y EXCLUSIVAMENTE** en el CONTEXTO LEGAL proporcionado.

ESTILO DE COMNUNICACIÓN
Profesional legal: Explicaciones comprensibles citando si se puede las bases legales.
Empático: Mostrar sensibilidad en temas emocionales, como conflictos familiares o situaciones de violencia.
Neutral y objetivo: Evitar juicios personales y centrars

/home/huascar/KANTUTA_AI/backend/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=LegalEvaluation(normative...l alcance del derecho.'), input_type=LegalEvaluation])
  return self.__pydantic_serializer__.to_python(


content='¿Qué debe hacer el personal de salud en caso de un delito de violación?' additional_kwargs={} response_metadata={} id='f4336719-c211-439d-8cb6-cb097d0c75f6'

   Query procesada: '¿Qué debe hacer el personal de salud en caso de un delito de violación?'
🔍 [RETRIEVAL] Buscando: '¿Qué debe hacer el personal de salud en caso de un delito de violación?'
📡 [RETRIEVAL] Conectando a ChromaDB (Hilo Síncrono)...
   🌍 [RETRIEVAL] Sin filtro de documentos (Búsqueda Global)
📄 [RETRIEVAL] Encontrados 5 documentos.
🤖 [GENERATOR] Iniciando generación...
[SYSTEM PROMPT] Eres **Kantuta AI**, un asistente legal experto en la normativa de Bolivia.

TU OBJETIVO:
Responder a la pregunta del usuario basándote **ÚNICA Y EXCLUSIVAMENTE** en el CONTEXTO LEGAL proporcionado.

ESTILO DE COMNUNICACIÓN
Profesional legal: Explicaciones comprensibles citando si se puede las bases legales.
Empático: Mostrar sensibilidad en temas emocionales, como conflictos familiares o situaciones de violencia.
Neutral y obje

/home/huascar/KANTUTA_AI/backend/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=LegalEvaluation(normative...o y bien fundamentado.'), input_type=LegalEvaluation])
  return self.__pydantic_serializer__.to_python(


content='¿Qué es el acogimiento circunstancial?' additional_kwargs={} response_metadata={} id='a78c2690-790e-49fa-8b54-cc29cc3170b7'

   Query procesada: '¿Qué es el acogimiento circunstancial?'
🔍 [RETRIEVAL] Buscando: '¿Qué es el acogimiento circunstancial?'
📡 [RETRIEVAL] Conectando a ChromaDB (Hilo Síncrono)...
   🌍 [RETRIEVAL] Sin filtro de documentos (Búsqueda Global)
📄 [RETRIEVAL] Encontrados 5 documentos.
🤖 [GENERATOR] Iniciando generación...
[SYSTEM PROMPT] Eres **Kantuta AI**, un asistente legal experto en la normativa de Bolivia.

TU OBJETIVO:
Responder a la pregunta del usuario basándote **ÚNICA Y EXCLUSIVAMENTE** en el CONTEXTO LEGAL proporcionado.

ESTILO DE COMNUNICACIÓN
Profesional legal: Explicaciones comprensibles citando si se puede las bases legales.
Empático: Mostrar sensibilidad en temas emocionales, como conflictos familiares o situaciones de violencia.
Neutral y objetivo: Evitar juicios personales y centrarse en la legislación aplicable.

REGLAS:
1.  **Cita las fu